In [ ]:
import psutil
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
import os
import gc

# ==========================================
# 1. SETUP E CONFIGURAZIONE
# ==========================================
print("RAM Iniziale:", psutil.virtual_memory())
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = True

In [ ]:

BASE_KAGGLE_PATH = "/kaggle/input/datasets/silvioliparoti/dpo-dataset-v2" 

DATASETS = [

    {
        "input": os.path.join(BASE_KAGGLE_PATH, "dataset_LPR_new_DPO_v2_2.csv"),
        "output_model": "deberta_judge_v2_2.pth"
    }
]

In [ ]:
# 2. CONFIGURAZIONE DEBERTA
MODEL_NAME = "microsoft/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4
GRAD_ACCUMULATION = 8 # Accumuliamo i gradienti per simulare un batch da 32, prima era 4
#Modifica degli iperparametri
#EPOCHS = 8  
#LR = 2e-5

#EPOCHS = 8
#LR = 1e-6

#continuiamo la grid search cercando una via di mezzo per aumentare le performance
# evitando l'overfitting/overalignement del primo test con 8 epoche
EPOCHS = 4 
LR = 2e-5


In [ ]:
# ==========================================
# 2. DEFINIZIONE CLASSI 
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class JokeDataset(Dataset):
    def __init__(self, df, tokenizer):
        col_text = 'jokeText' if 'jokeText' in df.columns else 'text'
        self.texts = df[col_text].values.astype(str)
        self.labels = df['Label'].values.astype(float)
        self.tokenizer = tokenizer
        
    def __len__(self): 
        return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], 
            truncation=True, 
            padding='max_length', 
            max_length=128, 
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }

class DebertaJudge(nn.Module):
    def __init__(self):
        super().__init__()
        # Forziamo il modello in float32 puro per evitare ogni problema di precisione
        self.bert = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
        self.bert.config.use_cache = False
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)
        
    def forward(self, input_ids, mask):
        out = self.bert(input_ids, attention_mask=mask)
        return self.regressor(out.last_hidden_state[:, 0, :])

In [ ]:
# ==========================================
# 3. CICLO DI TRAINING (PURO, SENZA AMP)
# ==========================================

from transformers import get_linear_schedule_with_warmup

for config in DATASETS:
    input_file = config["input"]
    output_model_name = config["output_model"]
    
    print("\n" + "="*60)
    print(f" AVVIO TRAINING PER: {output_model_name}")
    print("="*60)
    
    if not os.path.exists(input_file):
        print(f" ERRORE: File {input_file} non trovato. Salto...")
        continue

    # FIX per le righe corrotte
    df = pd.read_csv(input_file, engine='python', on_bad_lines='skip')
    df['Label'] = df['LPR_new'] 
    
    train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
    
    train_loader = DataLoader(JokeDataset(train_df, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(JokeDataset(val_df, tokenizer), batch_size=BATCH_SIZE)

    model = DebertaJudge().to(DEVICE)
    
    # QUI C'È IL TUO WEIGHT DECAY
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01) 
    criterion = nn.MSELoss()
    
    # -------------------------------------------------------------
    # === STEP 3 AGIUNTO QUI: CREAZIONE DELLO SCHEDULER ===
    # Lo calcoliamo qui perché abbiamo appena creato l'optimizer e il train_loader
    total_steps = (len(train_loader) // GRAD_ACCUMULATION) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps= int(total_steps * 0.1) , 
        num_training_steps=total_steps
    )
    # -------------------------------------------------------------

    # === INIZIO DEL NUOVO BLOCCO DA COPIARE ===
    history_train_loss = []
    history_val_loss = []
    
    for epoch in range(EPOCHS):
        # --- FASE DI TRAINING ---
        model.train()
        total_loss = 0
        
        for i, batch in enumerate(train_loader):
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbl = batch['labels'].clone().detach().to(DEVICE).float().unsqueeze(1)
            
            optimizer.zero_grad()
            
            preds = model(ids, mask)
            loss = criterion(preds, lbl)
            loss = loss / GRAD_ACCUMULATION
            
            loss.backward()
            
            # -------------------------------------------------------------
            # === STEP 4a AGGIUNTO QUI: GRADIENT CLIPPING ===
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            # -------------------------------------------------------------
            
            if (i + 1) % GRAD_ACCUMULATION == 0:
                optimizer.step()
                
                # -------------------------------------------------------------
                # === STEP 4b AGGIUNTO QUI: LO SCHEDULER FA IL SUO PASSO ===
                scheduler.step()
                # -------------------------------------------------------------
                
                optimizer.zero_grad()
            
            total_loss += loss.item() * GRAD_ACCUMULATION
            
            if i % 500 == 0 and i > 0:
                print(f"   Step {i}/{len(train_loader)}...")

        avg_train_loss = total_loss / len(train_loader)
        history_train_loss.append(avg_train_loss)
        
        # --- FASE DI VALIDATION (LA NOVITÀ) ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                lbl = batch['labels'].clone().detach().to(DEVICE).float().unsqueeze(1)
                
                preds = model(ids, mask)
                v_loss = criterion(preds, lbl)
                val_loss += v_loss.item()
                
        avg_val_loss = val_loss / len(val_loader)
        history_val_loss.append(avg_val_loss)

        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- DISEGNO DEL GRAFICO PER LA TESI ---
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, EPOCHS+1), history_train_loss, label='Train Loss', color='blue', marker='o', linewidth=2)
    plt.plot(range(1, EPOCHS+1), history_val_loss, label='Validation Loss', color='red', marker='s', linewidth=2)
    plt.title(f'Learning Curve: {output_model_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Epoche')
    plt.ylabel('MSE Loss')
    plt.legend()
    
    # Salva automaticamente il grafico come PNG
    nome_grafico = output_model_name.replace(".pth", ".png")
    plt.savefig(nome_grafico, dpi=300, bbox_inches='tight')
    plt.show()

    # Salvataggio del modello
    torch.save(model.state_dict(), output_model_name)
    print(f" Modello salvato: {output_model_name}")
    # === FINE DEL NUOVO BLOCCO ===